## From Information Retrieval to Streaming Response 📝

In this notebook I eant to extend the workflow introduced in **Week 2 — Day 5**.

Previously, we learned how to retrieve relevant information from a company website by:
1. Downloading the main webpage.
2. Extracting internal links.
3. Selecting the most relevant pages.
4. Aggregating their content.

In this notebook we build on that idea and introduce two important improvements:
- a **second model call with a different objective**
- a **streaming visualization of the model response** using a typewriter-like effect.

In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI

In [ ]:
# Initialize
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# Model
MODEL = 'gpt-5-nano'
openai = OpenAI()

# Part 1 — Retrieving Relevant Information from a Website 📥

Given a company website URL, the system:

1. Extracts internal links from the HTML.
2. Filters the most relevant links (e.g. *About*, *Products*, *Mission*, etc.).
3. Builds a prompt that aggregates this information.

This step allows us to **collect structured context directly from the website**, which can then be used by the language model to generate useful outputs.

This approach illustrates a common pattern in modern LLM applications:

> **Retrieve → Aggregate → Generate**

where the model receives curated context rather than raw input.

---

In [ ]:
# one-shot prompting application
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links to retrive information about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

# Part 2 — Generating an info overview 🌈

The second part of the notebook introduces a **new objective**. We now ask the model to act as an **assistant that produces a concise overview**.

To achieve this, we define two prompts:

### System Prompt
The **system prompt** defines the role of the model.
In this case, the model is instructed to behave as an assistant that extracts and organizes key information about a company.

### User Prompt
The **user prompt** contains:
- the company name
- the website URL
- the extracted content from relevant pages

The model then synthesizes this information into a structured company presentation.

This demonstrates an important principle in LLM applications:

> The **same retrieved data** can be used for **different tasks**, depending on how the prompts are designed.

---

In [ ]:
info_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short info about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
def get_info_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short info of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

# Streaming the Model Response 🪄

A new feature introduced in this notebook is **streaming the model output**. Instead of waiting for the full completion, the response is received **token by token** and displayed progressively.

To implement this behavior, we use:

- the `stream=True` parameter in the OpenAI API
- `IPython.display` utilities

In [ ]:
from IPython.display import display, Markdown, update_display

def stream_info(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": info_system_prompt},
            {"role": "user", "content": get_info_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
stream_info("Datapizza", "https://datapizza.tech/it/")

### Why Streaming Matters ﹖
Streaming responses improves the user experience in several ways:

- Lower perceived latency
Users see text appearing immediately instead of waiting for the full response.

- Better interactivity
The interface feels more responsive and conversational.

- Closer to real-world applications
Many AI assistants and chat systems rely on streaming outputs.

---